# 07 Target Resume Assembly

Stage 7 assembles a targeted resume from the canonical resume, selected evidence, and resume positioning.

Inputs:
- `artifacts/canonical_master_resume.json`
- `artifacts/selected_evidence.json`
- `artifacts/capability_review.json`
- `artifacts/resume_positioning.json`
- `artifacts/target_archetype.json`

Outputs:
- `artifacts/target_resume_v1.json`
- `artifacts/target_resume_trace_v1.json`

This stage creates a canonical resume artifact suitable for rendering. It uses LLM judgment only to convert selected evidence into concise, targeted experience bullets. It does not invent new claims, employers, dates, technologies, or outcomes.

In [1]:
import sys, pathlib

# Locate repo root by looking for the genai_demos resume_builder package and notebooks/resume-builder
p = pathlib.Path().resolve()
root = None
for d in (p,) + tuple(p.parents):
    if (d / 'src' / 'genai_demos' / 'resume_builder').is_dir() and (d / 'notebooks' / 'resume-builder').is_dir():
        root = d
        break
if root is None:
    raise RuntimeError('Could not find repository root containing src/genai_demos/resume_builder and notebooks/resume-builder')
src_path = root / 'src'
src_str = str(src_path)
if src_str not in sys.path:
    sys.path.insert(0, src_str)
print('Repo root:', str(root))
print('Added src to sys.path:', src_str)


Repo root: /Users/douglasdaly/GitHub/Generative-AI
Added src to sys.path: /Users/douglasdaly/GitHub/Generative-AI/src


In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import json
from collections import defaultdict

from genai_demos.resume_builder.config import ARTIFACT_DIR
from genai_demos.resume_builder.helpers import load_json, save_json
from genai_demos.resume_builder.capabilities import call_json_model

load_dotenv()

MODEL = ChatOpenAI(
    model="gpt-4.1",
    temperature=0,
)


## 7A. Load inputs

In [3]:
canonical_resume = load_json(ARTIFACT_DIR / "canonical_master_resume.json")
selected_evidence = load_json(ARTIFACT_DIR / "selected_evidence.json")
capability_review = load_json(ARTIFACT_DIR / "capability_review.json")
resume_positioning = load_json(ARTIFACT_DIR / "resume_positioning.json")
target_archetype = load_json(ARTIFACT_DIR / "target_archetype.json")

canonical_resume.keys(), selected_evidence.keys(), resume_positioning.keys()

(dict_keys(['schema_version', 'header', 'sections']),
 dict_keys(['selection_summary', 'selected_evidence', 'excluded_evidence']),
 dict_keys(['positioning_summary', 'target_title', 'summary', 'core_expertise', 'positioning_notes', 'core_expertise_resume_block', 'core_expertise_resume_block_original']))

## 7B. Build Summary and Core Expertise Sections

In [4]:
def build_summary_section(resume_positioning):
    return {
        "section_id": "SUMMARY",
        "type": "paragraph",
        "content": [
            resume_positioning["summary"]["text"]
        ],
    }


def build_core_expertise_section(resume_positioning):
    block = resume_positioning["core_expertise_resume_block"]

    return {
        "section_id": "EXPERTISE",
        "type": "bullet",
        "content": [
            f"{item['label']} - {item['text']}"
            for item in block.get("items", [])
        ],
    }


summary_section = build_summary_section(resume_positioning)
core_expertise_section = build_core_expertise_section(resume_positioning)

summary_section, core_expertise_section

({'section_id': 'SUMMARY',
  'type': 'paragraph',
  'content': ['Senior technical leader with deep expertise architecting enterprise AI/ML platforms, governed GenAI and tool-calling systems, and operational intelligence solutions. Built governed analytics assistants, monitoring frameworks, and reusable data/AI platforms with strong emphasis on controls, observability, and auditable execution. Experienced across cloud-native AI infrastructure, large-scale data platforms, executive alignment, and technical mentorship. Known for translating ambiguous business needs into scalable, reliable analytical systems that improve decision-making and operational performance.']},
 {'section_id': 'EXPERTISE',
  'type': 'bullet',
  'content': ['AI Platform Architecture - enterprise AI platforms, governed analytics assistants, operational intelligence systems, reusable AI/analytics frameworks',
   'GenAI & Tool-Calling Systems - LLM applications, structured outputs, tool orchestration, RAG/GraphRAG conc

## 7C. Prepare Evidence for Experience Generation

In [5]:
EXPERIENCE_SOURCE_TIERS = {
    "must_include",
    "strong_include",
    "optional",
}

experience_evidence = [
    item
    for item in selected_evidence["selected_evidence"]
    if item["selection_tier"] in EXPERIENCE_SOURCE_TIERS
]

len(experience_evidence)

23

In [6]:
def get_professional_experience_section(canonical_resume):
    matches = [
        section
        for section in canonical_resume.get("sections", [])
        if section.get("heading", "").lower() == "professional experience"
    ]

    if not matches:
        raise ValueError("Could not find Professional Experience section")

    return matches[0]


def build_experience_source_order_map(canonical_resume):
    """
    Build source-order lookup from the canonical resume.

    This preserves the human-approved reverse chronological ordering from the
    canonical resume instead of sorting by evidence score.

    Returns:
    - by_group_key: maps Stage 7 group keys to canonical order tuples
    - by_source_label: fallback map using source_label
    """
    prof_exp = get_professional_experience_section(canonical_resume)

    by_group_key = {}
    by_source_label = {}

    for exp_idx, exp in enumerate(prof_exp.get("content", [])):
        organization = exp.get("organization", "")
        role = exp.get("role", "")
        dates = exp.get("dates", "")

        direct_source_label = " | ".join(
            x for x in [organization, role] if x
        )

        direct_group_key = (
            organization,
            role,
            dates,
            "",
            direct_source_label,
        )

        by_group_key[direct_group_key] = (exp_idx, 0)
        by_source_label[direct_source_label] = (exp_idx, 0)

        for block_idx, block in enumerate(exp.get("content", []), start=1):
            if not isinstance(block, dict):
                continue

            label = block.get("label", "")
            block_dates = block.get("dates", dates)

            source_label = " | ".join(
                x for x in [organization, role, label] if x
            )

            group_key = (
                organization,
                role,
                block_dates,
                label,
                source_label,
            )

            by_group_key[group_key] = (exp_idx, block_idx)
            by_source_label[source_label] = (exp_idx, block_idx)

    return by_group_key, by_source_label

In [7]:
def group_evidence_by_source(evidence_items, canonical_resume):
    """
    Group selected evidence by resume source while preserving canonical
    reverse-chronological order.

    Evidence rank is used only within a source group, not to order resume jobs.
    """
    source_order_by_group_key, source_order_by_label = build_experience_source_order_map(
        canonical_resume
    )

    grouped = defaultdict(list)

    for item in evidence_items:
        key = (
            item.get("organization", ""),
            item.get("role", ""),
            item.get("dates", ""),
            item.get("label", ""),
            item.get("source_label", ""),
        )
        grouped[key].append(item)

    groups = []

    for key, items in grouped.items():
        organization, role, dates, label, source_label = key

        source_order = source_order_by_group_key.get(
            key,
            source_order_by_label.get(source_label, (999, 999))
        )

        groups.append({
            "organization": organization,
            "role": role,
            "dates": dates,
            "label": label,
            "source_label": source_label,
            "source_order": source_order,
            "evidence_items": sorted(
                items,
                key=lambda x: x.get("rank", 999)
            ),
        })

    return sorted(
        groups,
        key=lambda g: g["source_order"]
    )

In [8]:
experience_groups = group_evidence_by_source(
    evidence_items=experience_evidence,
    canonical_resume=canonical_resume,
)

for group in experience_groups:
    print(group["source_order"], group["source_label"])

(0, 1) Daly Engineers LLC | Principal Consultant | Consolidated Edison (via ABC Consulting)
(0, 2) Daly Engineers LLC | Principal Consultant | NBCUniversal (via Apex Systems)
(1, 0) SimpliLearn | AI Instructor
(2, 0) Interview Kickstart | AI Instructor
(3, 0) Intuit (via The Cydio Group) | Data Scientist
(4, 0) Meta | Data Scientist
(5, 0) Samsung Electronics (via Harvey Nash) | Data Scientist
(6, 0) Facteus | Head of Data Science
(7, 0) Nike (via Intersoft Inc.) | Expert SEO Solutions Architect
(8, 0) Capital One | Senior Manager, Operations Analysis
(9, 0) Raytheon | Senior Principal Systems Engineer / Program Manager


In [9]:
for group in experience_groups:
    print("\n" + "=" * 100)
    print(group["source_label"])
    for item in group["evidence_items"]:
        print(f"- {item['selection_tier']} | rank {item['rank']} | {item['evidence_text']}")


Daly Engineers LLC | Principal Consultant | Consolidated Edison (via ABC Consulting)
- must_include | rank 2 | Built a governed analytics assistant using Gemini, BigQuery, and tool-calling architectures that applied analyst rules and operational controls while maintaining predictable and auditable execution.
- strong_include | rank 7 | Developed automated profiling, type inference, duplicate detection, candidate key analysis, and cross-system join validation frameworks across 100+ source tables and millions of operational and financial records.
- optional | rank 19 | Developed a phased analytical maturity roadmap adopted as the foundation for anomaly detection, construction analytics, and AI-enabled decision support initiatives.
- optional | rank 21 | Led analytical readiness assessment for Oracle/EBS and Maximo data migrated to BigQuery, establishing the foundation for cross-domain analytics across construction operations and financial systems.

Daly Engineers LLC | Principal Consult

## 7D. Generate Targeted Experience Bullets

In [10]:
def build_experience_bullets_prompt(group, target_archetype):
    expected = {
        "source_label": group["source_label"],
        "role": group["role"],
        "organization": group["organization"],
        "dates": group["dates"],
        "label": group["label"],
        "bullets": [
            {
                "text": "string",
                "supporting_evidence_ids": ["string"],
                "cautions": ["string"]
            }
        ]
    }

    compact_evidence = [
        {
            "evidence_id": item["evidence_id"],
            "selection_tier": item["selection_tier"],
            "evidence_text": item["evidence_text"],
            "primary_supporting_signals": item.get("primary_supporting_signals", []),
            "cautions": item.get("cautions", []),
        }
        for item in group["evidence_items"]
    ]

    return f"""
You are generating targeted resume bullets for one resume role/source group.

Target archetype:
{target_archetype.get("title", "")}

Archetype summary:
{target_archetype.get("archetype_summary", "")}

Role/source group:
{json.dumps({
    "source_label": group["source_label"],
    "organization": group["organization"],
    "role": group["role"],
    "dates": group["dates"],
    "label": group["label"],
}, indent=2)}

Selected evidence:
{json.dumps(compact_evidence, indent=2)}

Task:
Generate concise targeted resume bullets for this role/source group.

Rules:
- Use only the provided evidence.
- Do not invent employers, dates, technologies, metrics, responsibilities, or outcomes.
- Do not overstate roadmap, assessment, teaching, or prototype evidence as production implementation.
- Preserve important cautions.
- Combine related evidence when appropriate.
- Prefer strong accomplishment bullets over keyword lists.
- Each bullet must cite supporting_evidence_ids.
- Generate 1 to 4 bullets depending on the amount and strength of evidence.
- Use plain, polished resume language.
- Do not use first person.
- Return valid JSON only.

Return exactly this structure:
{json.dumps(expected, indent=2)}
"""

In [11]:
experience_bullet_groups = []

for idx, group in enumerate(experience_groups, start=1):
    print(f"{idx}/{len(experience_groups)}: {group['source_label']}")

    prompt = build_experience_bullets_prompt(
        group=group,
        target_archetype=target_archetype,
    )

    result = call_json_model(prompt, MODEL)
    experience_bullet_groups.append(result)

len(experience_bullet_groups)

1/11: Daly Engineers LLC | Principal Consultant | Consolidated Edison (via ABC Consulting)
2/11: Daly Engineers LLC | Principal Consultant | NBCUniversal (via Apex Systems)
3/11: SimpliLearn | AI Instructor
4/11: Interview Kickstart | AI Instructor
5/11: Intuit (via The Cydio Group) | Data Scientist
6/11: Meta | Data Scientist
7/11: Samsung Electronics (via Harvey Nash) | Data Scientist
8/11: Facteus | Head of Data Science
9/11: Nike (via Intersoft Inc.) | Expert SEO Solutions Architect
10/11: Capital One | Senior Manager, Operations Analysis
11/11: Raytheon | Senior Principal Systems Engineer / Program Manager


11

## 7E. Validate Generated Bullets

In [12]:
valid_evidence_ids = {
    item["evidence_id"]
    for item in experience_evidence
}


def validate_experience_bullet_groups(experience_bullet_groups, valid_evidence_ids):
    errors = []

    for group_idx, group in enumerate(experience_bullet_groups):
        bullets = group.get("bullets", [])

        if not bullets:
            errors.append(f"group[{group_idx}] has no bullets: {group.get('source_label')}")

        if len(bullets) > 4:
            errors.append(f"group[{group_idx}] has more than 4 bullets")

        for bullet_idx, bullet in enumerate(bullets):
            text = bullet.get("text", "")

            if not text:
                errors.append(f"group[{group_idx}].bullets[{bullet_idx}] missing text")

            evidence_ids = bullet.get("supporting_evidence_ids", [])

            if not evidence_ids:
                errors.append(
                    f"group[{group_idx}].bullets[{bullet_idx}] has no supporting_evidence_ids"
                )

            for evidence_id in evidence_ids:
                if evidence_id not in valid_evidence_ids:
                    errors.append(
                        f"group[{group_idx}].bullets[{bullet_idx}] unknown evidence_id: {evidence_id}"
                    )

    return errors


experience_bullet_errors = validate_experience_bullet_groups(
    experience_bullet_groups,
    valid_evidence_ids,
)

experience_bullet_errors

[]

## 7F. Build Professional Experience Section

This is where we need to preserve the canonical schema style.

For grouped client work, I’d create experience entries with subsections.

In [13]:
def build_professional_experience_section(experience_bullet_groups):
    """
    Assemble generated bullets into a renderer-safe Professional Experience section.

    Renderer-safe rules:
    - Professional Experience section has type "experience".
    - Each experience entry is a dict with role, organization, dates, type, content.
    - If an experience entry has direct role bullets, use type "bullet" and content as strings.
    - If an experience entry has labeled client/project blocks, use type "subsections"
      and content as subsection dictionaries.
    - Never place raw string bullets directly inside a "subsections" content list.

    This fixes the Stage 7 bug where direct role bullets were added as raw strings
    under an experience entry marked as type "subsections".
    """
    entries_by_role = {}

    for group in experience_bullet_groups:
        role_key = (
            group.get("organization", ""),
            group.get("role", ""),
            group.get("dates", ""),
        )

        if role_key not in entries_by_role:
            organization, role, dates = role_key
            entries_by_role[role_key] = {
                "role": role,
                "organization": organization,
                "dates": dates,
                "direct_bullets": [],
                "subsections": [],
                "source_order": group.get("source_order", (999, 999)),
            }

        bullets = [
            bullet["text"]
            for bullet in group.get("bullets", [])
            if bullet.get("text")
        ]

        label = group.get("label", "")

        if label:
            entries_by_role[role_key]["subsections"].append({
                "label": label,
                "type": "bullet",
                "content": bullets,
            })
        else:
            entries_by_role[role_key]["direct_bullets"].extend(bullets)

    finalized_entries = []

    for entry in entries_by_role.values():
        direct_bullets = entry.pop("direct_bullets")
        subsections = entry.pop("subsections")
        source_order = entry.pop("source_order", (999, 999))

        if subsections and direct_bullets:
            entry["type"] = "subsections"
            entry["content"] = [
                {
                    "label": "Role Highlights",
                    "type": "bullet",
                    "content": direct_bullets,
                },
                *subsections,
            ]

        elif subsections:
            entry["type"] = "subsections"
            entry["content"] = subsections

        else:
            entry["type"] = "bullet"
            entry["content"] = direct_bullets

        entry["_source_order"] = source_order
        finalized_entries.append(entry)

    finalized_entries = sorted(
        finalized_entries,
        key=lambda x: x.get("_source_order", (999, 999))
    )

    for entry in finalized_entries:
        entry.pop("_source_order", None)

    return {
        "section_id": "EXPERIENCE",
        "type": "experience",
        "content": finalized_entries,
    }

professional_experience_section = build_professional_experience_section(
    experience_bullet_groups
)

professional_experience_section

{'section_id': 'EXPERIENCE',
 'type': 'experience',
 'content': [{'role': 'Principal Consultant',
   'organization': 'Daly Engineers LLC',
   'dates': 'Jan 2026 - Present',
   'type': 'subsections',
   'content': [{'label': 'Consolidated Edison (via ABC Consulting)',
     'type': 'bullet',
     'content': ['Architected and delivered a governed analytics assistant leveraging Gemini, BigQuery, and tool-calling architectures, embedding analyst rules and operational controls to ensure predictable, auditable, and compliant execution of generative AI workflows.',
      'Developed automated frameworks for profiling, type inference, duplicate detection, candidate key analysis, and cross-system join validation across 100+ source tables and millions of operational and financial records, enhancing data quality and interoperability.',
      'Defined a phased analytical maturity roadmap adopted as the foundation for anomaly detection, construction analytics, and AI-enabled decision support initiati

In [14]:
# Validation - should return []
def find_renderer_schema_issues(target_resume_or_section):
    """
    Check for renderer schema issues before saving/rendering.

    Accepts either the full target resume or just the Professional Experience section.
    """
    if target_resume_or_section.get("type") == "experience":
        sections = [target_resume_or_section]
    else:
        sections = target_resume_or_section.get("sections", [])

    issues = []

    for section_idx, section in enumerate(sections):
        if section.get("type") != "experience":
            continue

        for exp_idx, exp in enumerate(section.get("content", [])):
            exp_type = exp.get("type")
            content = exp.get("content", [])

            if exp_type == "subsections":
                for item_idx, item in enumerate(content):
                    if not isinstance(item, dict):
                        issues.append({
                            "section_idx": section_idx,
                            "experience_idx": exp_idx,
                            "item_idx": item_idx,
                            "role": exp.get("role"),
                            "organization": exp.get("organization"),
                            "issue": "subsections content item is not a dict",
                            "bad_item_type": type(item).__name__,
                        })

            elif exp_type == "bullet":
                for item_idx, item in enumerate(content):
                    if not isinstance(item, str):
                        issues.append({
                            "section_idx": section_idx,
                            "experience_idx": exp_idx,
                            "item_idx": item_idx,
                            "role": exp.get("role"),
                            "organization": exp.get("organization"),
                            "issue": "bullet content item is not a string",
                            "bad_item_type": type(item).__name__,
                        })

            else:
                issues.append({
                    "section_idx": section_idx,
                    "experience_idx": exp_idx,
                    "role": exp.get("role"),
                    "organization": exp.get("organization"),
                    "issue": f"unexpected experience entry type: {exp_type}",
                })

    return issues


find_renderer_schema_issues(professional_experience_section)

[]

## 7G. Add Education / Other Sections from Canonical Resume

For now, copy stable sections from the canonical resume. Do not regenerate them.

In [17]:
from copy import deepcopy

COPIED_SECTIONS = [
    {"section_id": "PROJECTS", "required": False},
    {"section_id": "EDUCATION", "required": True},
    {"section_id": "RECOGNITION", "required": True},
]

def normalize_section_id(value: str) -> str:
    return str(value).strip().upper()

def with_section_id(section: dict, section_id: str) -> dict:
    section = dict(section)

    existing = section.get("section_id")
    if existing and existing != section_id:
        raise ValueError(
            f"Section already has section_id={existing!r}, expected {section_id!r}"
        )

    section["section_id"] = section_id
    return section

def copy_sections_by_id(
    canonical_resume: dict,
    copied_section_config: list[dict],
    *,
    include_optional_sections: bool = True,
) -> list[dict]:
    sections_by_id = {}

    for section in canonical_resume.get("sections", []):
        section_id = section.get("section_id")

        # Do not fail here. Some canonical sections may be replaced by
        # generated target sections in this notebook.
        if not section_id:
            continue

        section_id = normalize_section_id(section_id)

        if section_id in sections_by_id:
            raise ValueError(f"Duplicate canonical section_id found: {section_id}")

        section = deepcopy(section)
        section["section_id"] = section_id
        sections_by_id[section_id] = section

    copied_sections = []

    for config in copied_section_config:
        section_id = normalize_section_id(config["section_id"])
        required = bool(config.get("required", True))

        if not required and not include_optional_sections:
            continue

        if section_id not in sections_by_id:
            if required:
                raise ValueError(f"Missing required canonical section_id: {section_id}")
            continue

        copied_sections.append(deepcopy(sections_by_id[section_id]))

    return copied_sections


print([
    {
        "heading": section.get("heading"),
        "section_id": section.get("section_id"),
        "type": section.get("type"),
    }
    for section in canonical_resume.get("sections", [])
])

copied_sections = copy_sections_by_id(
    canonical_resume,
    COPIED_SECTIONS,
    include_optional_sections=True,
)

content_sections = [
    with_section_id(summary_section, "SUMMARY"),
    with_section_id(core_expertise_section, "EXPERTISE"),
    with_section_id(professional_experience_section, "EXPERIENCE"),
    *copied_sections,
]

[{'heading': 'Summary', 'section_id': None, 'type': 'paragraph'}, {'heading': 'Core Technologies', 'section_id': None, 'type': 'subsections'}, {'heading': 'Professional Experience', 'section_id': None, 'type': 'experience'}, {'heading': 'Selected Projects', 'section_id': None, 'type': 'subsections'}, {'heading': 'Education', 'section_id': None, 'type': 'subsections'}, {'heading': 'Patents & Recognition', 'section_id': None, 'type': 'bullet'}]


ValueError: Missing required canonical section_id: EDUCATION

## 7H. Assemble Target Resume

In [ ]:


def normalize_section_id(value: str) -> str:
    return str(value).strip().upper()


def with_section_id(section: dict, section_id: str) -> dict:
    section = deepcopy(section)
    normalized_id = normalize_section_id(section_id)

    existing_id = section.get("section_id")
    if existing_id and normalize_section_id(existing_id) != normalized_id:
        raise ValueError(
            f"Section already has section_id={existing_id!r}, "
            f"but expected {normalized_id!r}"
        )

    section["section_id"] = normalized_id
    return section


def validate_sections_have_ids(sections: list[dict], source_name: str) -> list[dict]:
    validated = []

    for section in sections or []:
        section = deepcopy(section)
        section_id = section.get("section_id")

        if not section_id:
            raise ValueError(
                f"{source_name} section is missing section_id: "
                f"heading={section.get('heading')!r}"
            )

        section["section_id"] = normalize_section_id(section_id)
        validated.append(section)

    return validated


def validate_unique_section_ids(sections: list[dict]) -> None:
    seen = set()

    for section in sections:
        section_id = normalize_section_id(section.get("section_id", ""))

        if not section_id:
            raise ValueError(f"Section missing section_id: {section.get('heading')!r}")

        if section_id in seen:
            raise ValueError(f"Duplicate section_id found: {section_id}")

        seen.add(section_id)


def assemble_target_resume_content(
    canonical_resume: dict,
    summary_section: dict,
    core_expertise_section: dict,
    professional_experience_section: dict,
    copied_sections: list[dict],
) -> dict:
    """
    Assemble the targeted resume content artifact.

    This stage handles content only. It does not decide final section order,
    display headings, target title, or rendering.
    """
    generated_sections = [
        with_section_id(summary_section, "SUMMARY"),
        with_section_id(core_expertise_section, "EXPERTISE"),
        with_section_id(professional_experience_section, "EXPERIENCE"),
    ]

    copied_sections = validate_sections_have_ids(
        copied_sections,
        source_name="copied_sections",
    )

    sections = [
        *generated_sections,
        *copied_sections,
    ]

    validate_unique_section_ids(sections)

    return {
        "schema_version": canonical_resume.get("schema_version", "1.0"),
        "header": deepcopy(canonical_resume.get("header", {})),
        "sections": sections,
    }


target_resume_content_v1 = assemble_target_resume_content(
    canonical_resume=canonical_resume,
    summary_section=summary_section,
    core_expertise_section=core_expertise_section,
    professional_experience_section=professional_experience_section,
    copied_sections=copied_sections,
)

print(target_resume_content_v1.keys())
[
    (
        idx,
        section.get("section_id"),
        section.get("heading"),
        section.get("type"),
    )
    for idx, section in enumerate(target_resume_content_v1["sections"])
]

Backfilled header.title from resume_positioning.target_title
dict_keys(['schema_version', 'header', 'sections'])


[(0, 'Summary', 'paragraph'),
 (1, 'CORE EXPERTISE', 'bullet'),
 (2, 'Professional Experience', 'experience'),
 (3, 'Education', 'subsections'),
 (4, 'Patents & Recognition', 'bullet'),
 (5, 'Selected Projects', 'subsections')]

## 7I. Build Traceability Artifact

In [ ]:
def build_target_resume_trace(
    resume_positioning,
    experience_bullet_groups,
    selected_evidence,
    target_archetype,
):
    return {
        "target_archetype": target_archetype.get("title", ""),
        "source_artifacts": [
            "canonical_master_resume.json",
            "selected_evidence.json",
            "capability_review.json",
            "resume_positioning.json",
        ],
        "summary_trace": {
            "supporting_capabilities": resume_positioning.get("summary", {}).get(
                "supporting_capabilities", []
            ),
            "supporting_evidence_ids": resume_positioning.get("summary", {}).get(
                "supporting_evidence_ids", []
            ),
            "cautions": resume_positioning.get("summary", {}).get("cautions", []),
        },
        "core_expertise_trace": resume_positioning.get(
            "core_expertise_resume_block", {}
        ).get("items", []),
        "experience_bullet_trace": experience_bullet_groups,
        "selection_summary": selected_evidence.get("selection_summary", {}),
    }


target_resume_trace_v1 = build_target_resume_trace(
    resume_positioning=resume_positioning,
    experience_bullet_groups=experience_bullet_groups,
    selected_evidence=selected_evidence,
    target_archetype=target_archetype,
)

## 7J. Save Artifacts

In [ ]:
save_json(target_resume_v1, ARTIFACT_DIR / "target_resume_content_v1.json")
save_json(target_resume_trace_v1, ARTIFACT_DIR / "target_resume_trace_v1.json")

## 7K. Human Review Display

In [ ]:
def display_subsection_block(block):
    """
    Display a canonical subsection block.

    Supports subsection content types used in the canonical resume:
    - bullet
    - paragraph
    - inline_list
    - string fallback
    """
    label = block.get("label", "")
    context = block.get("context", "")
    dates = block.get("dates", "")
    block_type = block.get("type", "")
    content = block.get("content", [])

    heading_parts = [x for x in [label, context, dates] if x]
    if heading_parts:
        print("\n" + " | ".join(heading_parts))

    if block_type == "bullet":
        for item in content:
            print(f"- {item}")

    elif block_type == "paragraph":
        for item in content:
            print(item)

    elif block_type == "inline_list":
        if isinstance(content, list):
            print(", ".join(str(x) for x in content))
        else:
            print(content)

    else:
        # Fallback for unexpected subsection formats.
        if isinstance(content, list):
            for item in content:
                print(f"- {item}")
        elif content:
            print(content)

def display_target_resume_text(target_resume):
    header = target_resume.get("header", {})

    print("=" * 100)
    print(header.get("name", ""))
    print("=" * 100)

    for section in target_resume.get("sections", []):
        print("\n" + "=" * 100)
        print(section.get("heading", "").upper())
        print("=" * 100)

        section_type = section.get("type")

        if section_type == "paragraph":
            for paragraph in section.get("content", []):
                print(paragraph)

        elif section_type == "bullet":
            for bullet in section.get("content", []):
                print(f"- {bullet}")

        elif section_type == "subsections":
            for block in section.get("content", []):
                if isinstance(block, dict):
                    display_subsection_block(block)
                else:
                    print(f"- {block}")

        elif section_type == "experience":
            for exp in section.get("content", []):
                print(
                    "\n"
                    + f"{exp.get('role', '')} | "
                    + f"{exp.get('organization', '')} | "
                    + f"{exp.get('dates', '')}"
                )

                for item in exp.get("content", []):
                    if isinstance(item, str):
                        print(f"- {item}")
                    elif isinstance(item, dict):
                        label = item.get("label", "")
                        if label:
                            print(f"\n{label}")
                        for bullet in item.get("content", []):
                            print(f"- {bullet}")

        else:
            print(f"[Unsupported section type: {section_type}]")
            print(section.get("content", ""))
            display_target_resume_text(target_resume_v1)

In [ ]:
display_target_resume_text(target_resume_v1)

Douglas Gordon Daly

SUMMARY
Senior technical leader with deep expertise architecting enterprise AI/ML platforms, governed GenAI and tool-calling systems, and operational intelligence solutions. Built governed analytics assistants, monitoring frameworks, and reusable data/AI platforms with strong emphasis on controls, observability, and auditable execution. Experienced across cloud-native AI infrastructure, large-scale data platforms, executive alignment, and technical mentorship. Known for translating ambiguous business needs into scalable, reliable analytical systems that improve decision-making and operational performance.

CORE EXPERTISE
- AI Platform Architecture - enterprise AI platforms, governed analytics assistants, operational intelligence systems, reusable AI/analytics frameworks
- GenAI & Tool-Calling Systems - LLM applications, structured outputs, tool orchestration, RAG/GraphRAG concepts, prompt engineering, evaluation, traceability
- Governance & Operational Controls - r

## Stage 7 Completion Summary

Stage 7 assembles a targeted resume JSON from the resume positioning layer, selected evidence, and canonical resume structure.

Outputs:

- `artifacts/target_resume_v1.json`
- `artifacts/target_resume_trace_v1.json`

Key decisions:

- Summary and Core Expertise are copied from `resume_positioning.json`.
- Professional Experience bullets are generated from selected evidence, with supporting evidence IDs retained in the trace artifact.
- Resume chronology follows the canonical resume order, not evidence score order.
- Education and Patents & Recognition are copied directly from the canonical resume.
- Selected Projects is treated as an optional section and excluded by default.
- Rendering is intentionally deferred to Stage 8.

Stage 7 is complete when:

- `target_resume_v1.json` follows the canonical resume schema.
- Professional Experience appears in reverse chronological order.
- Stable sections display correctly.
- `target_resume_trace_v1.json` preserves evidence traceability for generated bullets.

In [ ]:
[(idx, section.get("heading"), section.get("type")) for idx, section in enumerate(target_resume_v1["sections"])]

[(0, 'Summary', 'paragraph'),
 (1, 'CORE EXPERTISE', 'bullet'),
 (2, 'Professional Experience', 'experience'),
 (3, 'Education', 'subsections'),
 (4, 'Patents & Recognition', 'bullet'),
 (5, 'Selected Projects', 'subsections')]